# TP N°1 ACN

Propon ́e de manera precisa una regla de comportamiento de pasajeros
Pens ́a en tu experiencia o busc ́a videos y responde: cu ́anto tarda en levantarse y dejar
lugar un pasajero sentado en el pasillo para que pase un pasajero que va a ventana?
Si una persona llega a su fila y su asiento est ́a vac ́ıo, cu ́anto tarde en sentarse? y si
tiene carry on?

Desde el momento en que un pasajero sube el avión, tiene un repertorio limitado de acciones:
- Avanzar.
- Levantarse.
- Guardar equipaje.
- Sentarse.

Asimismo, pueden adoptar algunos de los siguientes estados:
- Posición actual.
- Asiento.
- Asiento conseguido.
- Tiene carry-on (a mano).
- Sentado.
- Pasajero delante (en pasillo central).
- Pasajero en pasillo (ya estando sentado en fila junto al pasillo).

Vamos a asumir las siguientes reglas de comportamiento para los pasajeros:
- El pasajero que llega a su asiento vacío y no posee carry-on se demora de 4 a 12 segundos en sentarse.
- El pasajero que llega a su asiento vacío y que sí posee carry-on se demora de 12 a 32 segundos en sentarse.
- Cada pasajero que debe levantarse para habilitar la llegada al asiento agrega de 3 a 5 segundos al tiempo total de onboarding.
- Si se requiere guardar el carry-on y esperar para sentarse, necesariamente se debe guardar el carry-on primero para luego esperar a que se le deje pasar.



In [ ]:
# Definimos agentes y modelo
import mesa
import random
import numpy
from mesa.discrete_space import CellAgent, OrthogonalMooreGrid, Grid2DMovingAgent

class passenger(Grid2DMovingAgent):
    def __init__(self, model, seat:tuple[int,int], carryon:bool):
        super().__init__(model)
        self.seat:tuple[int,int] = seat
        self.carryon:bool = carryon
        self.arrived_at_seat:bool = False
        self.is_active = False

    def activate(self):
        self.is_active = True
        print("Agente del asiento", self.seat, " ingresa al avión.")

    def walk_forward(self):
        print("Agente del asiento", self.seat, " avanza a ", self.cell)

class plane(mesa.Model):
    
    '''
    Para seleccionar el método de llenado, se tienen cuatro opciones:
    - 'btf' para la política Back-to-Front.
    - 'rand' para la política aleatoria.
    - 'wilma' para la política Window-Middle-Aisle.
    - 'stfn' para el método de Steffen.
    '''

    def __init__(self, n=100, p=0.5, onboarding_method='rand'):
        super().__init__()
        self.queue:list[tuple[int,int]] = list()

        # Espacio físico
        self.plane_grid = OrthogonalMooreGrid((25,5), capacity=1)

        # Tiempo
        self.time = 0

        # Generamos la queue
        if onboarding_method == 'btf' or onboarding_method == 'rand':

            for row in range(25):
                for col in range(5):
                    if (col != 2): 
                        self.queue.append((row,col))

            random.shuffle(self.queue)

            if onboarding_method == 'btf':
                self.queue.sort(key=lambda tup: tup[0], reverse=True)

        elif onboarding_method == 'wilma':

            for row in range(25):
                for col in range(5):
                    if (col != 2): 
                        self.queue.append((row,col))
            
            random.shuffle(self.queue)
            self.queue.sort(key=lambda tup: tup[1] % 2 == 0, reverse=True)
                        
        elif onboarding_method == 'stfn':

            for col in [0,4]:
                for row in range(24,-1,-2):
                    self.queue.append((row,col))
            
            for col in [0,4]:
                for row in range(23,0,-2):
                    self.queue.append((row,col))

            for col in [1,3]:
                for row in range(24,-1,-2):
                    self.queue.append((row,col))
            
            for col in [1,3]:
                for row in range(23,0,-2):
                    self.queue.append((row,col))
        
        else:
            #agregar error
            pass

        # Creamos lista de agentes
        passengers = passenger.create_agents(
            model=self, 
            n=n, 
            seat=self.queue, 
            carryon = random.choices([0,1], [p, 1-p])[0]
        )  

    def step(self):
        self.agents.select(lambda a: a.unique_id == 1).do("activate")
        self.agents.select(lambda a: a.unique_id == 1).do("walk_forward") 
        self.time = self.time + 1

In [17]:
model = plane(onboarding_method='stfn')
for i in range(1):
    model.step(i)

c:\Users\camil_kzug21e\Documents\Ditella\2026\ACN\TP1\ACN-TP1\.venv\Lib\site-packages\mesa\discrete_space\grid.py:104: UserWarning: Random number generator not specified, this can make models non-reproducible. Please pass a random number generator explicitly
  super().__init__(capacity=capacity, random=random, cell_klass=cell_klass)


TypeError: Model._wrapped_step() takes 1 positional argument but 2 were given